In [0]:
%run "../SetUp/setup" 

### Access Azure Data Lake using Service Principal
**Steps to follow:**
1. Register Azure AD Application/ Service Principal
2. Generate a secret/ password for the application
3. Set spark config with App/ Client Id, Directory/ Tenant Id & Secret
4. Assign role "Storage Blob Data Contributor" to the Data Lake

path,name,size,modificationTime
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-21/,2021-03-21/,0,1768733801000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-28/,2021-03-28/,0,1768733591000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-04-18/,2021-04-18/,0,1768733721000


[FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/__unitystorage/', name='__unitystorage/', size=0, modificationTime=1769227742000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta/', name='drivers_convert_to_delta/', size=0, modificationTime=1769359802000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta_new/', name='drivers_convert_to_delta_new/', size=0, modificationTime=1769360049000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_external/', name='results_external/', size=0, modificationTime=1769228062000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_partitioned/', name='results_partitioned/', size=0, modificationTime=1769228706000)]

In [0]:
%run "../Includes/configs" 

In [0]:
%run "../Includes/comm_func" 

**Produce Constructor Standings**

In [0]:
dbutils.widgets.text("p_file_date", "")

In [0]:
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
race_results_df = spark.read.format("delta").load(f"{presentation_folder_path}/race_results")

In [0]:
from pyspark.sql.functions import sum, count, when, col, lit

In [0]:
constructor_standings_df = race_results_df.groupBy("race_year", "team").agg(sum("points").alias("total_points"), count(when(col("position") == 1, True)).alias("wins"))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import desc, rank, asc 

In [0]:
constructor_rank_spec = Window.partitionBy("race_year").orderBy(desc("total_points"), desc("wins"))
final_df = constructor_standings_df.withColumn("rank", rank().over(constructor_rank_spec)).withColumn("file_date", lit(v_file_date))

In [0]:
display(final_df)

race_year,team,total_points,wins,rank,file_date
1950,Alfa Romeo,267.0,18,1,2021-04-18
1950,Talbot-Lago,60.0,0,2,2021-04-18
1950,Ferrari,54.0,0,3,2021-04-18
1950,Kurtis Kraft,39.0,3,4,2021-04-18
1950,Maserati,33.0,0,5,2021-04-18
1950,Deidt,30.0,0,6,2021-04-18
1950,Simca,9.0,0,7,2021-04-18
1950,Milano,0.0,0,8,2021-04-18
1950,ERA,0.0,0,8,2021-04-18
1950,Cooper,0.0,0,8,2021-04-18


In [0]:
final_df_dedup = final_df.dropDuplicates(
    ["race_year", "team"]
)

In [0]:
final_df_dedup.write \
    .mode("append") \
    .format("delta") \
    .partitionBy("race_year") \
    .save(f"{presentation_folder_path}/constructor_standings")

In [0]:
spark.read.format("delta").load(f"{presentation_folder_path}/constructor_standings") \
    .select("file_date") \
    .distinct() \
    .show(truncate=False)

+----------+
|file_date |
+----------+
|2021-03-21|
|2021-04-18|
|2021-03-28|
+----------+



In [0]:
#final_df.write.mode("overwrite").parquet(f"{presentation_folder_path}/constructor_standings")

In [0]:
merge_condition = "tgt.team = src.team AND tgt.race_year = src.race_year"
merge_delta_data(final_df_dedup, 'f1_presentation', 'constructor_standings', presentation_folder_path, merge_condition, 'race_year')

In [0]:
%sql
SELECT * FROM f1_presentation.constructor_standings
ORDER BY race_year DESC;

race_year,team,total_points,wins,rank,file_date
2021,Mercedes,120.0,2,1,2021-04-18
2021,Red Bull,106.0,2,2,2021-04-18
2021,McLaren,82.0,0,3,2021-04-18
2021,Ferrari,68.0,0,4,2021-04-18
2021,AlphaTauri,16.0,0,5,2021-04-18
2021,Aston Martin,10.0,0,6,2021-04-18
2021,Alpine F1 Team,6.0,0,7,2021-04-18
2021,Alfa Romeo,0.0,0,8,2021-04-18
2021,Williams,0.0,0,8,2021-04-18
2021,Haas F1 Team,0.0,0,8,2021-04-18
